# evaluate

Executable notebook module. English explanations precede each functional block. Restart the kernel after changing modules.


In [ ]:
from pathlib import Path
import json, sys, types
PROJECT = Path.cwd()
assert (PROJECT / 'app.ipynb').is_file(), 'Open this notebook from EcoSort-Demo.'

def load_notebook(name):
    if name in sys.modules:
        return sys.modules[name]
    path = PROJECT / (name + '.ipynb')
    notebook = json.loads(path.read_text())
    module = types.ModuleType(name)
    module.__file__ = str(path)
    sys.modules[name] = module
    try:
        for dependency in notebook['metadata'].get('dependencies', []):
            load_notebook(dependency)
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code' and 'module' in cell.get('metadata', {}).get('tags', []):
                exec(compile(''.join(cell['source']), str(path), 'exec'), module.__dict__)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_notebook('app')
load_notebook('train')
load_notebook('disposal_rules')
__file__ = str(PROJECT / 'evaluate.ipynb')


## Imports and configuration

Load required libraries and define the paths, labels and constants used by the following functions.


In [ ]:
"""Re-evaluate the project-trained model on the fixed validation and test splits."""
import hashlib
import json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import gradio as gr
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from app import BASE, EcoSort
from train import dataset, load_manifest, LABELS
from disposal_rules import disposal_guidance




## main

Execute this module workflow. Read the ordered steps and parameters below before enabling execution.


In [ ]:
def main():
    manifest_path = BASE / "data/split_manifest.csv"
    manifest = load_manifest()
    audit = json.loads((BASE / "data/dataset_verification.json").read_text())
    assert len(manifest) == audit["retained_images"] == 147303
    assert manifest.sha256.is_unique and manifest.groupby("group_id").split.nunique().max() == 1  # Check exact uniqueness and group separation across splits.
    assert manifest.groupby("split").size().to_dict() == audit["split_counts"]
    assert all(Path(p).is_file() for p in manifest.path)
    engine = EcoSort()
    assert engine.classifier.classes == LABELS  # Ensure training and deployment use the same output-label order.
    for bad in (None, "not an image"):
        try:
            engine.classify(bad)
            raise AssertionError("Invalid image accepted.")
        except gr.Error:
            pass
    assert "red" in disposal_guidance("plastic", "Soft plastic bag or wrapper")[0]
    assert "drop-off" in disposal_guidance("battery")[0]
    assert "Potentially" in disposal_guidance("plastic")[0]
    metrics = {}
    for split in ("validation", "test"):
        frame = manifest[manifest.split == split].sort_values(["label", "path"]).reset_index(drop=True)
        truth = frame.label.map(dict(zip(LABELS, range(len(LABELS))))).to_numpy()
        probabilities = engine.classifier.model.predict(dataset(frame), verbose=1)  # Run batch inference with the deployed checkpoint.
        assert probabilities.shape == (len(frame), len(LABELS)) and np.isfinite(probabilities).all()
        predicted = probabilities.argmax(axis=1)
        metrics[split] = dict(images=len(frame), accuracy=float(accuracy_score(truth, predicted)),  # Record image count and overall classification accuracy.
                              macro_f1=float(f1_score(truth, predicted, average="macro", zero_division=0)))  # Record an equal-class macro-F1 summary.
        print(split, metrics[split], flush=True)
        # Confirm deployment preprocessing agrees with batch evaluation.
        for index in (0, len(frame)//2, len(frame)-1):
            with Image.open(frame.path.iloc[index]) as image:
                scores, decision, guidance = engine.classify(image)
                single = engine.classifier.probabilities(image)
                assert int(single.argmax()) == int(predicted[index])  # Require its winning class to match batch evaluation.
                assert np.isclose(sum(scores.values()), 1)
                assert "SigLIP" not in decision
                assert "yellow lid bin" not in decision or "Potentially" in decision
                confirmed = engine.classify(image, "Soft plastic bag or wrapper")
                assert "red lid" in confirmed[1]  # Require that confirmation to produce the expected red-bin guidance.
    target = BASE / "model/results.json"
    record = json.loads(target.read_text())
    record["evaluation"] = dict(
        verified_at=datetime.now(ZoneInfo("Australia/Sydney")).isoformat(),
        metrics=metrics,
        rules_and_inputs_passed=True,
        manifest_sha256=hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
        limitations="Previously observed dataset splits, not fresh independent validation. No real-world accuracy guarantee.",
    )

    target.parent.mkdir(exist_ok=True)
    temporary = target.with_suffix(".tmp")
    temporary.write_text(json.dumps(record, indent=2))
    temporary.replace(target)  # Atomically replace the old record after writing succeeds.
    print(json.dumps(record, indent=2), flush=True)


## Optional execution

Disabled by default to avoid starting expensive work or overwriting outputs during inspection.


In [ ]:
RUN_WORKFLOW = False
if RUN_WORKFLOW:
    saved_arguments = sys.argv
    try:
        sys.argv = ['evaluate']
        main()
    finally:
        sys.argv = saved_arguments
